# NBA Player Dashboard

Provides a summary view for a given NBA player using the [`nba_api`](https://github.com/swar/nba_api) library.

**Sections:**
1. Player Lookup
2. Last 5 Game Averages
3. Last 10 Game Averages
4. Season Averages
5. Team Injury Report
6. Summary Dashboard

> **Rate Limit:** nba_api calls NBA.com stats endpoints. A 0.6s sleep is added between requests to avoid throttling.

In [ ]:
# Install dependencies if needed
# !pip install nba_api pandas requests

In [ ]:
import time
import requests
import pandas as pd
from IPython.display import display, HTML

from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import (
    playergamelog,
    playercareerstats,
    commonplayerinfo,
    commonteamroster,
    playerdashboardbylastngames,
    playerdashboardbygamesplits,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 120)

SLEEP     = 0.6   # seconds between API calls
THRESHOLD = 3.0   # pt diff (1H vs 2H) to label a hot/cold pattern

## Configuration

Set `PLAYER_NAME` to the player you want to look up and `SEASON` to the desired season string (e.g. `"2024-25"`).

In [11]:
PLAYER_NAME = "Duncan Robinson"   # <-- Change me
SEASON      = "2025-26"        # <-- NBA season format: YYYY-YY

## 1. Player Lookup

Resolves the player name to an NBA player ID used by all subsequent API calls.

In [12]:
def find_player(name: str) -> dict:
    """Find a player by full name. Raises ValueError if not found."""
    matches = players.find_players_by_full_name(name)
    if not matches:
        raise ValueError(f"No player found matching '{name}'. Check spelling or try a partial name.")
    if len(matches) > 1:
        print(f"Multiple matches found — using first result. All matches:")
        for m in matches:
            print(f"  {m['full_name']} (id={m['id']}, active={m['is_active']})")
    return matches[0]


player_info = find_player(PLAYER_NAME)
PLAYER_ID   = player_info["id"]

print(f"Player : {player_info['full_name']}")
print(f"ID     : {PLAYER_ID}")
print(f"Active : {player_info['is_active']}")

Player : Duncan Robinson
ID     : 1629130
Active : True


## 2 & 3. Last 5 / Last 10 Game Averages

Uses `PlayerDashboardByLastNGames` to pull pre-computed rolling averages directly from NBA.com, then also fetches the full game log for the per-game table.

In [ ]:
# PlayerDashboardByLastNGames returns 6 DataFrames in API response order.
# expected_data is defined alphabetically in the source, but get_data_frames()
# reflects the actual NBA Stats API result set order (numerical by N):
#
#   Index 0 → GameNumberPlayerDashboard
#   Index 1 → Last5PlayerDashboard
#   Index 2 → Last10PlayerDashboard
#   Index 3 → Last15PlayerDashboard
#   Index 4 → Last20PlayerDashboard
#   Index 5 → OverallPlayerDashboard
#
_LAST_N_IDX = {5: 1, 10: 2, 15: 3, 20: 4}

def get_all_last_n_dashboards(player_id: int, season: str) -> dict:
    """
    Single API call → returns a dict of raw DataFrames keyed by span (5/10/15/20).
    Reads the correct index for each window based on actual API response order.
    """
    dash = playerdashboardbylastngames.PlayerDashboardByLastNGames(
        player_id=player_id,
        season=season,
    )
    time.sleep(SLEEP)
    dfs = dash.get_data_frames()
    return {n: dfs[idx] for n, idx in _LAST_N_IDX.items()}


# Counting stats returned as totals — must divide by GP
_COUNT_STATS = ["MIN", "PTS", "REB", "AST", "STL", "BLK",
                "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "PLUS_MINUS"]

_RENAME_MAP = {
    "MIN": "MPG", "PTS": "PPG", "REB": "RPG", "AST": "APG",
    "STL": "SPG", "BLK": "BPG", "FGM": "FGM/G", "FGA": "FGA/G",
    "FG3M": "3PM/G", "FG3A": "3PA/G", "FTM": "FTM/G", "FTA": "FTA/G",
    "PLUS_MINUS": "+/-",
}

def to_per_game(df: pd.DataFrame) -> pd.DataFrame:
    """Divide counting stats by GP to produce per-game averages, then rename columns."""
    out = df.copy()
    gp  = max(out["GP"].iloc[0], 1) if "GP" in out.columns and not out.empty else 1
    for col in _COUNT_STATS:
        if col in out.columns:
            out[col] = (out[col] / gp).round(1)
    for col in ["FG_PCT", "FG3_PCT", "FT_PCT"]:
        if col in out.columns:
            out[col] = out[col].round(3)
    return out.rename(columns=_RENAME_MAP)


DISPLAY_COLS = [
    "GP", "MPG", "PPG", "RPG", "APG", "SPG", "BPG",
    "FGM/G", "FGA/G", "FG_PCT",
    "3PM/G", "3PA/G", "FG3_PCT",
    "FTM/G", "FTA/G", "FT_PCT",
    "+/-",
]

# One API call, two windows
all_last_n = get_all_last_n_dashboards(PLAYER_ID, SEASON)
last5_df   = to_per_game(all_last_n[5])
last10_df  = to_per_game(all_last_n[10])

def show_rolling_avg(df: pd.DataFrame, label: str):
    cols = [c for c in DISPLAY_COLS if c in df.columns]
    print(f"\n{'=' * 55}")
    print(f" {label}")
    print(f"{'=' * 55}")
    display(df[cols].reset_index(drop=True))

show_rolling_avg(last5_df,  f"Last 5 Game Averages — {player_info['full_name']} ({SEASON})")
show_rolling_avg(last10_df, f"Last 10 Game Averages — {player_info['full_name']} ({SEASON})")

### Per-Game Log (Last 10)

Full game-by-game breakdown for the most recent 10 games.

In [14]:
def get_game_log(player_id: int, season: str) -> pd.DataFrame:
    """Fetch the full season game log, returned newest-first."""
    gl = playergamelog.PlayerGameLog(player_id=player_id, season=season)
    time.sleep(SLEEP)
    df = gl.get_data_frames()[0]

    # MIN is stored as "MM:SS" — convert to float
    if df["MIN"].dtype == object:
        df["MIN"] = df["MIN"].apply(
            lambda x: float(x.split(":")[0]) + float(x.split(":")[1]) / 60
            if ":" in str(x) else float(x)
        ).round(1)

    return df


games_df = get_game_log(PLAYER_ID, SEASON)

LOG_COLS = ["GAME_DATE", "MATCHUP", "WL", "MIN", "PTS", "REB", "AST",
            "STL", "BLK", "FG_PCT", "FG3_PCT", "FT_PCT", "PLUS_MINUS"]

log_display = games_df[[c for c in LOG_COLS if c in games_df.columns]]

print(f"\nGame Log (Last 10) — {player_info['full_name']} ({SEASON})")
display(log_display.head(10).reset_index(drop=True))


Game Log (Last 10) — Duncan Robinson (2025-26)


,GAME_DATE,MATCHUP,WL,MIN,PTS,REB,AST,STL,BLK,FG_PCT,FG3_PCT,FT_PCT,PLUS_MINUS
0,"Feb 21, 2026",DET @ CHI,W,24,17,3,2,0,0,0.42,0.50,1.00,11
1,"Feb 19, 2026",DET @ NYK,W,21,9,1,0,0,0,0.50,0.50,0.00,3
2,"Feb 11, 2026",DET @ TOR,W,25,13,4,1,1,1,0.50,0.40,0.75,17
3,"Feb 09, 2026",DET @ CHA,W,24,18,3,4,0,0,0.80,0.33,0.50,27
4,"Feb 06, 2026",DET vs. NYK,W,20,9,4,2,1,0,0.38,0.43,0.00,7
5,"Feb 05, 2026",DET vs. WAS,L,30,21,1,2,0,0,0.40,0.40,1.00,-7
6,"Feb 03, 2026",DET vs. DEN,W,25,20,3,2,0,0,0.58,0.67,0.00,-1
7,"Feb 01, 2026",DET vs. BKN,W,22,8,3,1,1,0,0.38,0.40,0.00,41
8,"Jan 30, 2026",DET @ GSW,W,30,15,2,2,0,0,0.46,0.50,0.00,-1
9,"Jan 29, 2026",DET @ PHX,L,30,9,1,1,1,0,0.36,0.17,0.00,-3


## 3.5 Quarter-by-Quarter Averages (Last 5 & 10 Games)

Uses **`PlayerDashboardByGameSplits`** — `ByPeriodPlayerDashboard` (index 2) with `per_mode_detailed="PerGame"`.

- **2 API calls total** (one for L5, one for L10) — no per-game box score loops needed
- Stats are pre-computed per-game averages; no division required
- `GROUP_VALUE` `"1"` / `"2"` / `"3"` / `"4"` = Q1 – Q4; OT rows are filtered out

**Patterns detected (based on L5):**

| Label | Condition |
|---|---|
| **Hot Start / Cold Finish** | Q1+Q2 avg ≥ Q3+Q4 avg by `THRESHOLD` pts |
| **Cold Start / Hot Finish** | Q3+Q4 avg ≥ Q1+Q2 avg by `THRESHOLD` pts |
| **Consistent** | Difference within `THRESHOLD` |

In [ ]:
def get_quarter_avgs(player_id: int, season: str, last_n: int) -> pd.DataFrame:
    """
    Return per-game averages by quarter (Q1–Q4) for the last N games.
    Uses PlayerDashboardByGameSplits / ByPeriodPlayerDashboard (index 2).
    per_mode_detailed='PerGame' means stats are already per-game — no division needed.
    """
    dash = playerdashboardbygamesplits.PlayerDashboardByGameSplits(
        player_id=player_id,
        season=season,
        last_n_games=last_n,
        per_mode_detailed="PerGame",
    )
    time.sleep(SLEEP)
    df = dash.get_data_frames()[2]  # ByPeriodPlayerDashboard

    # Keep Q1–Q4 only (GROUP_VALUE "1"–"4"); drop OT periods if present
    df = df[df["GROUP_VALUE"].astype(str).isin(["1", "2", "3", "4"])].copy()
    df["Quarter"] = "Q" + df["GROUP_VALUE"].astype(str)

    keep = [c for c in [
        "Quarter", "GP", "PTS", "FGM", "FGA", "FG_PCT",
        "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",
        "REB", "AST", "STL", "BLK", "PLUS_MINUS",
    ] if c in df.columns]

    return df[keep].reset_index(drop=True)


def detect_pattern(quarter_df: pd.DataFrame) -> dict:
    """Detect hot/cold/consistent scoring pattern from a quarter-avgs DataFrame."""
    avgs = {r["Quarter"]: round(r["PTS"], 1) for _, r in quarter_df.iterrows()}
    avg_q1, avg_q2 = avgs.get("Q1", 0), avgs.get("Q2", 0)
    avg_q3, avg_q4 = avgs.get("Q3", 0), avgs.get("Q4", 0)
    avg_1h = avg_q1 + avg_q2
    avg_2h = avg_q3 + avg_q4
    diff   = avg_1h - avg_2h

    if diff >= THRESHOLD:
        label  = "HOT START / COLD FINISH"
        detail = f"Scores {diff:.1f} more pts/g in the 1st half — fades as game goes on."
    elif diff <= -THRESHOLD:
        label  = "COLD START / HOT FINISH"
        detail = f"Scores {abs(diff):.1f} more pts/g in the 2nd half — heats up late."
    else:
        label  = "CONSISTENT"
        detail = "Scoring is fairly even across all four quarters."

    return {
        "label"  : label, "detail" : detail,
        "avg_1h" : avg_1h, "avg_2h" : avg_2h,
        "avg_q1" : avg_q1, "avg_q2" : avg_q2,
        "avg_q3" : avg_q3, "avg_q4" : avg_q4,
        "best_q" : max(avgs, key=avgs.get),
        "worst_q": min(avgs, key=avgs.get),
    }


# ── Fetch ─────────────────────────────────────────────────────────────────────
qbq_last5  = get_quarter_avgs(PLAYER_ID, SEASON, 5)
qbq_last10 = get_quarter_avgs(PLAYER_ID, SEASON, 10)
pattern    = detect_pattern(qbq_last5)

# ── Side-by-side comparison table ─────────────────────────────────────────────
STAT_COLS = ["GP", "PTS", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
             "FTM", "FTA", "FT_PCT", "REB", "AST", "STL", "BLK", "PLUS_MINUS"]

def build_qbq_display(qdf: pd.DataFrame, label: str) -> pd.DataFrame:
    """Pivot quarter rows into a single labelled display DataFrame."""
    cols = [c for c in STAT_COLS if c in qdf.columns]
    out  = qdf[["Quarter"] + cols].set_index("Quarter")
    out.index.name = label
    return out.round(2)

print(f"\nQuarter Averages (per game) — {player_info['full_name']} ({SEASON})")

print(f"\n{'─'*40}  Last 5 Games")
display(build_qbq_display(qbq_last5, "L5 · Quarter"))

print(f"\n{'─'*40}  Last 10 Games")
display(build_qbq_display(qbq_last10, "L10 · Quarter"))

# ── 1H / 2H summary ──────────────────────────────────────────────────────────
def half_summary(qdf: pd.DataFrame, label: str) -> pd.DataFrame:
    """Aggregate Q1+Q2 and Q3+Q4 into a first-half / second-half row."""
    pts = {r["Quarter"]: r["PTS"] for _, r in qdf.iterrows()}
    return pd.DataFrame([
        {label: "1st Half (Q1+Q2)", "PTS/G": round(pts.get("Q1",0) + pts.get("Q2",0), 1)},
        {label: "2nd Half (Q3+Q4)", "PTS/G": round(pts.get("Q3",0) + pts.get("Q4",0), 1)},
    ]).set_index(label)

halves = half_summary(qbq_last5, "L5").join(half_summary(qbq_last10, "L10"), lsuffix=" (L5)", rsuffix=" (L10)")
print(f"\nHalf Scoring — {player_info['full_name']}")
display(halves)

# ── Pattern ───────────────────────────────────────────────────────────────────
print(f"\n{'─' * 52}")
print(f"  SCORING PATTERN  (based on last 5 games)")
print(f"{'─' * 52}")
print(f"  Q1 {pattern['avg_q1']:.1f}  |  Q2 {pattern['avg_q2']:.1f}  |  "
      f"Q3 {pattern['avg_q3']:.1f}  |  Q4 {pattern['avg_q4']:.1f}")
print(f"  1st Half: {pattern['avg_1h']:.1f} pts/g   2nd Half: {pattern['avg_2h']:.1f} pts/g")
print(f"{'─' * 52}")
print(f"  Pattern  →  {pattern['label']}")
print(f"  {pattern['detail']}")
print(f"  Best Q   :  {pattern['best_q']}  |  Worst Q: {pattern['worst_q']}")
print(f"{'─' * 52}")

## 4. Season Averages

Pulls career stats and computes per-game averages for the configured season.

In [15]:
def get_season_averages(player_id: int, season: str) -> pd.Series:
    """Return per-game averages for the given season from career stats."""
    career = playercareerstats.PlayerCareerStats(player_id=player_id)
    time.sleep(SLEEP)

    totals_df = career.get_data_frames()[0]  # SeasonTotalsRegularSeason

    season_row = totals_df[totals_df["SEASON_ID"] == season]
    if season_row.empty:
        print(f"[warn] No data for {season}, showing most recent season instead.")
        season_row = totals_df.tail(1)

    row = season_row.iloc[0]
    gp  = max(row["GP"], 1)  # guard against divide-by-zero

    return pd.Series({
        "Season"   : row["SEASON_ID"],
        "Team"     : row.get("TEAM_ABBREVIATION", ""),
        "GP"       : int(row["GP"]),
        "MPG"      : round(row["MIN"]  / gp, 1),
        "PPG"      : round(row["PTS"]  / gp, 1),
        "RPG"      : round(row["REB"]  / gp, 1),
        "APG"      : round(row["AST"]  / gp, 1),
        "SPG"      : round(row["STL"]  / gp, 1),
        "BPG"      : round(row["BLK"]  / gp, 1),
        "FG%"      : round(row["FG_PCT"]  * 100, 1),
        "3P%"      : round(row["FG3_PCT"] * 100, 1),
        "FT%"      : round(row["FT_PCT"]  * 100, 1),
    })


season_avgs = get_season_averages(PLAYER_ID, SEASON)

print(f"\nSeason Averages — {player_info['full_name']} ({SEASON})")
display(season_avgs.to_frame(name="Value").T.reset_index(drop=True))


Season Averages — Duncan Robinson (2025-26)


,Season,Team,GP,MPG,PPG,RPG,APG,SPG,BPG,FG%,3P%,FT%
0,2025-26,DET,53,28.00,12.30,2.70,1.80,0.60,0.30,44.20,40.50,74.70


## 5. Team Injury Report

Resolves the player's current team then fetches injury data from the ESPN public API (no key required).  
The roster is displayed alongside injury statuses so you can see which teammates are out or questionable going into the next game.

> **Note:** `nba_api` does not expose a native injury endpoint. ESPN's public API is used as a reliable substitute.

In [16]:
def get_player_team(player_id: int) -> dict:
    """Return basic team info for the player's current team."""
    info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
    time.sleep(SLEEP)
    row = info.get_data_frames()[0].iloc[0]
    return {
        "team_id"   : int(row["TEAM_ID"]),
        "team_name" : row["TEAM_NAME"],
        "team_abbr" : row["TEAM_ABBREVIATION"],
    }


team_info = get_player_team(PLAYER_ID)
print(f"{player_info['full_name']} — {team_info['team_name']} ({team_info['team_abbr']})")

Duncan Robinson — Pistons (DET)


In [17]:
def get_team_roster(team_id: int, season: str) -> pd.DataFrame:
    """Return the current team roster."""
    roster = commonteamroster.CommonTeamRoster(team_id=team_id, season=season)
    time.sleep(SLEEP)
    df = roster.get_data_frames()[0]
    keep = [c for c in ["PLAYER", "NUM", "POSITION", "HEIGHT", "WEIGHT", "AGE", "EXP"] if c in df.columns]
    return df[keep].reset_index(drop=True)


roster_df = get_team_roster(team_info["team_id"], SEASON)
print(f"\n{team_info['team_name']} Roster ({SEASON})")
display(roster_df)


Pistons Roster (2025-26)


,PLAYER,NUM,POSITION,HEIGHT,WEIGHT,AGE,EXP
0,Jalen Duren,0,C,6-10,250,22.00,3
1,Cade Cunningham,2,G,6-6,220,24.00,4
2,Isaac Jones,3,F,6-8,245,25.00,1
3,Ronald Holland II,5,F,6-8,206,20.00,1
4,Paul Reed,7,F,6-9,210,26.00,5
5,Caris LeVert,8,G,6-7,205,31.00,9
6,Ausar Thompson,9,G-F,6-7,205,23.00,2
7,Tobias Harris,12,F,6-8,226,33.00,14
8,Wendell Moore Jr.,14,G,6-5,215,24.00,3
9,Chaz Lanier,20,G,6-3,206,24.00,R


In [18]:
def get_espn_injuries(team_abbr: str) -> pd.DataFrame:
    """
    Fetch injury report from the ESPN public API.
    Endpoint: https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{abbr}/injuries
    """
    url = (
        f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba"
        f"/teams/{team_abbr.lower()}/injuries"
    )
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    rows = []
    for item in data.get("injuries", []):
        athlete = item.get("athlete", {})
        rows.append({
            "Player"  : athlete.get("displayName", "Unknown"),
            "Position": athlete.get("position", {}).get("abbreviation", ""),
            "Status"  : item.get("status", "Unknown"),
            "Comment" : item.get("longComment") or item.get("shortComment", ""),
            "Date"    : item.get("date", "")[:10] if item.get("date") else "",
        })

    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Player", "Position", "Status", "Comment", "Date"])


try:
    injury_df = get_espn_injuries(team_info["team_abbr"])
    if injury_df.empty:
        print(f"No injuries currently listed for {team_info['team_name']}.")
    else:
        print(f"\n{team_info['team_name']} Injury Report")
        display(injury_df)
except Exception as e:
    print(f"Could not fetch ESPN injury data: {e}")
    print("Fallback: https://www.espn.com/nba/injuries")

No injuries currently listed for Pistons.


## 6. Summary Dashboard

Consolidated view of all the above sections.

In [ ]:
SEP = "=" * 60

print(SEP)
print(f"  NBA PLAYER DASHBOARD")
print(f"  {player_info['full_name'].upper()}")
print(f"  {team_info['team_name']}  |  Season: {SEASON}")
print(SEP)

# ── Per-game averages comparison ──────────────────────────────────────────────
def _rolling_row(df: pd.DataFrame) -> dict:
    r = df.iloc[0]
    return {
        "GP": r.get("GP"), "MPG": r.get("MPG"),
        "PPG": r.get("PPG"), "RPG": r.get("RPG"), "APG": r.get("APG"),
        "SPG": r.get("SPG"), "BPG": r.get("BPG"),
        "FG%": round(r.get("FG_PCT",  0) * 100, 1),
        "3P%": round(r.get("FG3_PCT", 0) * 100, 1),
        "FT%": round(r.get("FT_PCT",  0) * 100, 1),
        "+/-": r.get("+/-"),
    }

comparison_df = pd.DataFrame([
    {"Span": "Last 5",             **_rolling_row(last5_df)},
    {"Span": "Last 10",            **_rolling_row(last10_df)},
    {"Span": f"Season ({SEASON})",
        "GP": season_avgs["GP"], "MPG": season_avgs["MPG"],
        "PPG": season_avgs["PPG"], "RPG": season_avgs["RPG"],
        "APG": season_avgs["APG"], "SPG": season_avgs["SPG"],
        "BPG": season_avgs["BPG"], "FG%": season_avgs["FG%"],
        "3P%": season_avgs["3P%"], "FT%": season_avgs["FT%"],
        "+/-": None,
    },
]).set_index("Span")

print("\n[ PER-GAME AVERAGES ]")
display(comparison_df)

# ── Quarter scoring averages ───────────────────────────────────────────────────
print("\n[ QUARTER SCORING AVERAGES  (PTS/G) ]")
try:
    def _qrow(qdf, label):
        pts = {r["Quarter"]: round(r["PTS"], 1) for _, r in qdf.iterrows()}
        q1, q2, q3, q4 = pts.get("Q1",0), pts.get("Q2",0), pts.get("Q3",0), pts.get("Q4",0)
        return {"Span": label, "Q1": q1, "Q2": q2, "1H": round(q1+q2,1),
                "Q3": q3, "Q4": q4, "2H": round(q3+q4,1),
                "Total": round(q1+q2+q3+q4,1)}

    qbq_cmp = pd.DataFrame([
        _qrow(qbq_last5,  "Last 5"),
        _qrow(qbq_last10, "Last 10"),
    ]).set_index("Span")
    display(qbq_cmp)

    print(f"\n  Pattern (L5)  →  {pattern['label']}")
    print(f"  {pattern['detail']}")
except NameError:
    print("  Quarter data unavailable — run section 3.5 first.")

# ── Recent game log ───────────────────────────────────────────────────────────
print("\n[ RECENT GAME LOG (Last 5) ]")
display(log_display.head(5).reset_index(drop=True))

# ── Injury report ─────────────────────────────────────────────────────────────
print(f"\n[ {team_info['team_name'].upper()} INJURY REPORT ]")
try:
    if injury_df.empty:
        print("  No injuries reported.")
    else:
        display(injury_df)
except NameError:
    print("  Injury data unavailable — run cell 5 first.")

print(f"\n{SEP}")